# Sequence-to-Sequence (Seq2Seq) with Attention

## Architecture Overview
- **Encoder**: Processes input sequence using LSTM, outputs all hidden states (not just final)
- **Attention**: Dynamically focuses on relevant encoder states at each decoder step
- **Decoder**: Uses attention context + previous token to generate output one token at a time

## Key Fixes from v1
1. **Multi-word output bug**: `tgt_indices[1:-1]` was cutting the last real word when EOS wasn't generated — fixed with conditional slicing
2. **Source padding during inference**: `translate_sentence` now pads input to match training length
3. **Encoder now returns all hidden states** for attention (not just final hidden/cell)
4. **Larger dataset** with more diverse sentences so the model generalizes better
5. **More training epochs** with learning rate scheduling

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import numpy as np
import random

## Data — expanded to help the model learn multi-word patterns

In [ ]:
# Expanded dataset — more pairs = better generalization
english_sentences = [
    "hello",
    "how are you",
    "good morning",
    "thank you",
    "good night",
    "good evening",
    "see you later",
    "i am fine",
    "what is your name",
    "nice to meet you",
]
french_sentences = [
    "bonjour",
    "comment ça va",
    "bon matin",
    "merci",
    "bonne nuit",
    "bonsoir",
    "à bientôt",
    "je vais bien",
    "quel est votre nom",
    "enchanté de vous rencontrer",
]

## Vocabulary & Tokenization

In [ ]:
def build_vocab(sentences):
    vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab


english_vocab = build_vocab(english_sentences)
french_vocab  = build_vocab(french_sentences)
french_idx2word = {v: k for k, v in french_vocab.items()}  # reverse lookup

print(f"English vocab size: {len(english_vocab)}")
print(f"French vocab size:  {len(french_vocab)}")


def tokenize(sentences, vocab, max_len):
    """
    Converts sentences to padded token index arrays.
    Format: [<SOS>, w1, w2, ..., <EOS>, <PAD>, <PAD>, ...]
    """
    tokenized = []
    for sentence in sentences:
        tokens = [vocab.get(w, vocab["<UNK>"]) for w in sentence.split()]
        tokens = [vocab["<SOS>"]] + tokens + [vocab["<EOS>"]]
        tokens += [vocab["<PAD>"]] * (max_len - len(tokens))
        tokenized.append(tokens)
    return np.array(tokenized)


max_len_eng = max(len(s.split()) for s in english_sentences) + 2
max_len_fr  = max(len(s.split()) for s in french_sentences) + 2

english_data = tokenize(english_sentences, english_vocab, max_len=max_len_eng)
french_data  = tokenize(french_sentences,  french_vocab,  max_len=max_len_fr)

print(f"Max source length: {max_len_eng}, Max target length: {max_len_fr}")

English vocab size: 25
French vocab size:  27
Max source length: 6, Max target length: 6


## Dataset & Model

In [17]:
class TranslationDataset(Dataset):
    def __init__(self, source_data, target_data):
        self.src = source_data
        self.tgt = target_data

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx], dtype=torch.long), \
               torch.tensor(self.tgt[idx], dtype=torch.long)


dataset   = TranslationDataset(english_data, french_data)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


# ── Encoder ────────────────────────────────────────────────────────────────
class Encoder(nn.Module):
    """
    LSTM encoder. Returns ALL hidden states (enc_outputs) so the attention
    mechanism can attend over the full input sequence.
    """
    def __init__(self, input_dim, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=0)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                 batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, x):
        embedded       = self.dropout(self.embedding(x))   # (B, src_len, embed)
        enc_out, (h, c) = self.lstm(embedded)              # enc_out: (B, src_len, hidden)
        return enc_out, h, c


# ── Attention ──────────────────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    """
    Additive (Bahdanau) attention.
    Score(s, h) = v · tanh(W_s · s + W_h · h)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_s  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_h  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v    = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        """
        decoder_hidden  : (B, hidden)  — top-layer decoder hidden state
        encoder_outputs : (B, src_len, hidden)
        returns context : (B, hidden)
        """
        src_len = encoder_outputs.size(1)
        # Expand decoder state to match every encoder step
        dec = decoder_hidden.unsqueeze(1).repeat(1, src_len, 1)  # (B, src_len, hidden)
        energy  = torch.tanh(self.W_s(dec) + self.W_h(encoder_outputs))  # (B, src_len, hidden)
        scores  = self.v(energy).squeeze(-1)                              # (B, src_len)
        weights = F.softmax(scores, dim=1)                                # (B, src_len)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs)        # (B, 1, hidden)
        return context.squeeze(1), weights


# ── Decoder ────────────────────────────────────────────────────────────────
class Decoder(nn.Module):
    """
    LSTM decoder with Bahdanau attention.
    Input to LSTM at each step = [embedding; attention_context]
    """
    def __init__(self, output_dim, embed_dim, hidden_dim, num_layers, dropout=0.3):
        super().__init__()
        self.embedding  = nn.Embedding(output_dim, embed_dim, padding_idx=0)
        self.attention  = BahdanauAttention(hidden_dim)
        # LSTM input = embed + context vector
        self.lstm       = nn.LSTM(embed_dim + hidden_dim, hidden_dim, num_layers,
                                  batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc         = nn.Linear(hidden_dim * 2, output_dim)  # hidden + context → vocab
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, hidden, cell, encoder_outputs):
        """
        x               : (B,)          — current input token index
        hidden / cell   : (layers, B, hidden)
        encoder_outputs : (B, src_len, hidden)
        """
        x        = x.unsqueeze(1)                           # (B, 1)
        embedded = self.dropout(self.embedding(x))          # (B, 1, embed)

        # Attention over all encoder steps
        top_hidden       = hidden[-1]                       # (B, hidden) — top LSTM layer
        context, weights = self.attention(top_hidden, encoder_outputs)  # (B, hidden)
        context_expanded = context.unsqueeze(1)            # (B, 1, hidden)

        lstm_input = torch.cat([embedded, context_expanded], dim=2)     # (B, 1, embed+hidden)
        out, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))     # out: (B, 1, hidden)

        out     = out.squeeze(1)                            # (B, hidden)
        pred    = self.fc(torch.cat([out, context], dim=1))# (B, output_dim)
        return pred, hidden, cell, weights


# ── Seq2Seq wrapper ────────────────────────────────────────────────────────
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        B, tgt_len       = tgt.size()
        tgt_vocab_size   = self.decoder.fc.out_features
        outputs          = torch.zeros(B, tgt_len, tgt_vocab_size).to(self.device)

        enc_out, hidden, cell = self.encoder(src)
        inp = tgt[:, 0]                        # first input = <SOS>

        for t in range(1, tgt_len):
            output, hidden, cell, _ = self.decoder(inp, hidden, cell, enc_out)
            outputs[:, t, :]        = output
            top1 = output.argmax(1)
            # Teacher forcing: sometimes feed the gold label, sometimes the prediction
            inp  = tgt[:, t] if random.random() < teacher_forcing_ratio else top1

        return outputs


# ── Instantiate ────────────────────────────────────────────────────────────
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim  = len(english_vocab)
output_dim = len(french_vocab)
embed_dim  = 128
hidden_dim = 256
num_layers = 2

encoder = Encoder(input_dim,  embed_dim, hidden_dim, num_layers)
decoder = Decoder(output_dim, embed_dim, hidden_dim, num_layers)
model   = Seq2Seq(encoder, decoder, device).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {total_params:,}")

Model parameters: 2,257,179


## Training

In [18]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Reduce LR when loss plateaus
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=french_vocab["<PAD>"])


def train(model, dataloader, optimizer, scheduler, criterion, device, num_epochs=20):
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0
        for src, tgt in dataloader:
            src, tgt = src.to(device), tgt.to(device)
            optimizer.zero_grad()

            output = model(src, tgt)                            # (B, tgt_len, vocab)
            output = output[:, 1:].reshape(-1, output.shape[2])# skip <SOS> prediction
            target = tgt[:, 1:].reshape(-1)                    # skip <SOS> in target

            loss = criterion(output, target)
            loss.backward()
            # Gradient clipping prevents exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(dataloader)
        scheduler.step(avg_loss)
        
        print(f"Epoch {epoch+1:3d}: Loss = {avg_loss:.4f}")


train(model, dataloader, optimizer, scheduler, criterion, device, num_epochs=10)

Epoch   1: Loss = 3.2771
Epoch   2: Loss = 3.0130
Epoch   3: Loss = 2.7788
Epoch   4: Loss = 2.5479
Epoch   5: Loss = 2.4620
Epoch   6: Loss = 2.2841
Epoch   7: Loss = 1.9947
Epoch   8: Loss = 1.8639
Epoch   9: Loss = 1.5916
Epoch  10: Loss = 1.2868


## Inference

### Fix applied here:
- Input is **padded to `max_len_eng`** (same as training) — prevents length mismatch
- `tgt_indices[1:-1]` is replaced with conditional slicing that **doesn't drop the last token** unless EOS is present

In [19]:
def translate_sentence(model, sentence, eng_vocab, fr_vocab, fr_idx2word,
                       max_len_eng, max_len_fr, device):
    model.eval()

    # ── Tokenise & pad source (must match training format) ──
    tokens = [eng_vocab.get(w, eng_vocab["<UNK>"]) for w in sentence.split()]
    tokens = [eng_vocab["<SOS>"]] + tokens + [eng_vocab["<EOS>"]]
    # Pad to max_len_eng so the encoder sees the same length it trained on
    tokens += [eng_vocab["<PAD>"]] * max(0, max_len_eng - len(tokens))
    src = torch.tensor(tokens[:max_len_eng]).unsqueeze(0).to(device)  # (1, max_len_eng)

    with torch.no_grad():
        enc_out, hidden, cell = model.encoder(src)

        generated = [fr_vocab["<SOS>"]]
        for _ in range(max_len_fr):
            inp = torch.tensor([generated[-1]]).to(device)             # (1,)
            out, hidden, cell, _ = model.decoder(inp, hidden, cell, enc_out)
            pred = out.argmax(1).item()
            generated.append(pred)
            if pred == fr_vocab["<EOS>"]:
                break

    # ── FIX: only strip trailing EOS; don't blindly drop the last token ──
    tokens_out = generated[1:]                  # drop leading <SOS>
    if tokens_out and tokens_out[-1] == fr_vocab["<EOS>"]:
        tokens_out = tokens_out[:-1]            # drop trailing <EOS> if present

    words = [fr_idx2word.get(i, "<UNK>") for i in tokens_out
             if i not in (fr_vocab["<PAD>"], fr_vocab["<UNK>"])]
    return " ".join(words)


# ── Test on all training sentences ──
print("=" * 45)
print(f"{'English':<25} {'French':>18}")
print("=" * 45)
for eng, expected_fr in zip(english_sentences, french_sentences):
    translation = translate_sentence(
        model, eng, english_vocab, french_vocab, french_idx2word,
        max_len_eng, max_len_fr, device
    )
    match = "✓" if translation == expected_fr else "✗"
    print(f"{match} {eng:<23} → {translation}")
    print(f"  (expected: {expected_fr})")

print(translate_sentence(
        model, "fine", english_vocab, french_vocab, french_idx2word,
        max_len_eng, max_len_fr, device
    ))

English                               French
✓ hello                   → bonjour
  (expected: bonjour)
✓ how are you             → comment ça va
  (expected: comment ça va)
✗ good morning            → bon
  (expected: bon matin)
✓ thank you               → merci
  (expected: merci)
✗ good night              → bonne
  (expected: bonne nuit)
✓ good evening            → bonsoir
  (expected: bonsoir)
✓ see you later           → à bientôt
  (expected: à bientôt)
✓ i am fine               → je vais bien
  (expected: je vais bien)
✗ what is your name       → quel est votre
  (expected: quel est votre nom)
✗ nice to meet you        → enchanté est
  (expected: enchanté de vous rencontrer)
je vais


In [20]:
print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(25, 128, padding_idx=0)
    (lstm): LSTM(128, 256, num_layers=2, batch_first=True, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(27, 128, padding_idx=0)
    (attention): BahdanauAttention(
      (W_s): Linear(in_features=256, out_features=256, bias=False)
      (W_h): Linear(in_features=256, out_features=256, bias=False)
      (v): Linear(in_features=256, out_features=1, bias=False)
    )
    (lstm): LSTM(384, 256, num_layers=2, batch_first=True, dropout=0.3)
    (fc): Linear(in_features=512, out_features=27, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)


AttributeError: 'Seq2Seq' object has no attribute 'dump'